# InsureAssist — LoRA Fine-Tuning (Phase 2)

**Goal:** teach a small open LLM to answer insurance questions in our style, cheaply.

**LoRA (Low-Rank Adaptation):** we freeze the big model and train only a tiny "adapter"
(a few MB). This runs on a **free Colab T4 GPU** in minutes.

### How to run
1. Open this notebook in Colab (it's in your public repo):
   **File → Open notebook → GitHub → `mzquadri/insureassist-rag-mlops` →
   `finetune/lora_finetune.ipynb`.**
2. **Runtime → Change runtime type → T4 GPU → Save.**
3. **Runtime → Run all.** When done, download the `adapter/` folder (Phase 3 uses it).

You'll learn: Hugging Face `transformers`, `peft` (LoRA), `trl` (SFTTrainer), and MLflow.

## 1. Install libraries (pinned versions that work together)

In [ ]:
!pip -q install "transformers==4.44.2" "peft==0.12.0" "trl==0.9.6" "datasets==2.20.0" \
    "accelerate==0.33.0" "bitsandbytes==0.43.1" "mlflow==2.16.2" 2>/dev/null
print("installed")

## 2. Check the GPU (must show a Tesla T4)

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
!nvidia-smi -L

## 3. Load the base model (4-bit) + tokenizer
**Phi-3-mini** is a small, open model (no license gate). 4-bit loading keeps it inside the
free GPU's memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True
)
print("Base model loaded.")

## 4. Training data → formatted text
We turn each (question, answer) pair into a chat and render it to a single `text` string
using the model's chat template. (In a real project you'd have thousands of examples;
this small set demonstrates the full workflow.)

In [ ]:
from datasets import Dataset

raw = [
    {"q": "Does home insurance cover water damage from a burst pipe?",
     "a": "Yes. Sudden and accidental water damage from a burst pipe is covered, including tracing and accessing the leak. Gradual leakage or wear and tear is not covered."},
    {"q": "What is the excess for an escape of water claim?",
     "a": "A higher excess of EUR 500 applies to escape-of-water claims, versus the standard EUR 250."},
    {"q": "Up to how much is jewellery covered per item?",
     "a": "Jewellery is covered up to a single-item limit of EUR 2,000 unless separately listed on the schedule."},
    {"q": "How long do I have to report a home claim?",
     "a": "Home insurance claims must be reported within 30 days of the incident."},
    {"q": "Does comprehensive auto cover include a courtesy car?",
     "a": "Yes, up to 14 days while your vehicle is repaired by an approved garage."},
    {"q": "What extra excess applies to drivers under 25?",
     "a": "An additional young-driver excess of EUR 300 on top of the EUR 400 own-damage excess."},
    {"q": "What is the maximum no-claims discount?",
     "a": "Up to 65%, reached after five claim-free years."},
    {"q": "Is mechanical breakdown covered by auto insurance?",
     "a": "No, mechanical or electrical breakdown is excluded."},
    {"q": "Is damage covered if the home is empty for two months?",
     "a": "No. Damage is excluded if the home is unoccupied for more than 60 consecutive days."},
    {"q": "What is needed before a car theft claim is processed?",
     "a": "A police report reference number is required."},
]

def to_text(ex):
    messages = [
        {"role": "system", "content": "You are a precise insurance policy assistant."},
        {"role": "user", "content": ex["q"]},
        {"role": "assistant", "content": ex["a"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

ds = Dataset.from_list([to_text(r) for r in raw])
print(ds)
print(ds[0]["text"][:400])

## 5. LoRA config
`r` is the adapter size; `target_modules` are the attention layers that get the adapter.
Only these tiny weights are trained — the base model stays frozen.

In [ ]:
from peft import LoraConfig

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["qkv_proj", "o_proj"],   # Phi-3 attention projection names
)

## 6. Train (TRL SFTTrainer) + log to MLflow
SFT = Supervised Fine-Tuning. MLflow records params, loss, and the adapter artifact so you
can compare runs — that's the experiment-tracking skill.

In [ ]:
import mlflow
from trl import SFTTrainer, SFTConfig

mlflow.set_experiment("insureassist-lora")

args = SFTConfig(
    output_dir="adapter",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    bf16=True,
    report_to=[],
    max_seq_length=1024,
    dataset_text_field="text",
)

with mlflow.start_run():
    mlflow.log_params({
        "base_model": BASE_MODEL, "r": lora.r, "lora_alpha": lora.lora_alpha,
        "epochs": args.num_train_epochs, "lr": args.learning_rate,
    })
    trainer = SFTTrainer(
        model=model, args=args, train_dataset=ds, peft_config=lora, tokenizer=tokenizer,
    )
    trainer.train()
    final_loss = trainer.state.log_history[-1].get("train_loss")
    if final_loss is not None:
        mlflow.log_metric("final_train_loss", final_loss)
    trainer.save_model("adapter")            # saves ONLY the small LoRA adapter
    mlflow.log_artifacts("adapter", artifact_path="lora_adapter")
    print("Training done. Adapter saved to ./adapter")

## 7. Quick test — does it answer in our style?

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)
messages = [
    {"role": "system", "content": "You are a precise insurance policy assistant."},
    {"role": "user", "content": "Is a burst pipe covered by home insurance?"},
]
out = pipe(messages, max_new_tokens=120, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

## 8. Download the adapter
In the Colab file browser (folder icon on the left), right-click the **`adapter`** folder →
**Download**. Put it in your repo at `finetune/adapter/` for Phase 3.

Optional — push to the Hugging Face Hub instead:
```python
# from huggingface_hub import login; login()
# trainer.model.push_to_hub("mzquadri/insureassist-phi3-lora")
```

**Next:** Phase 3 — plug this adapter into the RAG pipeline and evaluate with RAGAS.